<b> Transform Orders Data - String to JSON 
1. Pre-process the JSON String to JSON Object
2. Transform JSON String to JSON Object
3. Write transformed data to the silver schema 

In [0]:
df_orders = spark.table('gizmobox.bronze.py_orders')
display(df_orders)

<b> 1. Pre-process the JSON String to fix the Data Quality Issues
[Regexp replace function](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/regexp_replace) 

In [0]:
from pyspark.sql import functions as f

df_fixed_orders = (
    df_orders.select(
        f.regexp_replace('Value', '"order_date": (\\d{4}-\\d{2}-\\d{2})', '"order_date": "$1"').alias("fixed_value")
    )
)
display(df_fixed_orders)

<b> 2. Transform JSON String to JSON Object
- Function [schema of json](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/schema_of_json)
- Function [from json](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/from_json) 

In [0]:
df_with_schema = (
    df_fixed_orders.select (
        f.schema_of_json(f.col("fixed_value").alias("schema"))
    )
)
display(df_with_schema.limit(1))

In [0]:
orders_schema = '''STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>'''

In [0]:
df_json_orders = (
    df_fixed_orders.select (
        f.from_json("fixed_value", orders_schema).alias("json_value")
    )
)
display(df_json_orders)

<b> 3. Write transformed data to the silver schema 

In [0]:
df_json_orders.writeTo('gizmobox.silver.py_orders').createOrReplace()

In [0]:
%sql
SELECT * FROM gizmobox.silver.py_orders